# 06 DSPy 自动 Prompt 优化

前提：Step 02-05 已建立评估标准（PC, CI, IDS 作为目标函数）+ 最优输出格式已确定。

用 DSPy 框架自动搜索最优 prompt 组合：
1. 定义 DSPy Signature（5-bin 输出）
2. 定义优化目标：最小化泄露 L，同时保持准确率
3. 使用 MIPROv2 优化器搜索
4. 与手工策略对比

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import dspy
import json
import re
import numpy as np
import pandas as pd
from tqdm import tqdm

from src.models import *
from src.llm_client import LLMClient
from src.news_loader import load_test_cases, load_counterfactual_variants
from src.masking import apply_masking
from src.metrics import prediction_consistency, confidence_invariance, input_dependency_score, composite_leakage_score

## 1. 配置 DSPy + 加载数据

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

# 配置 DSPy 使用 DeepSeek（OpenAI 兼容接口）
lm = dspy.LM(
    model="openai/deepseek-chat",
    api_base="https://api.deepseek.com/v1",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    temperature=0.0,
    max_tokens=2048,
)
dspy.configure(lm=lm)

# 加载数据
test_cases = load_test_cases()
variants = load_counterfactual_variants()
variant_map = {}
for v in variants:
    variant_map.setdefault(v.original_case_id, {})[v.variant_type] = v

# 训练/测试集划分（前 70% 训练，后 30% 测试）
split = int(len(test_cases) * 0.7)
train_cases = test_cases[:split]
eval_cases = test_cases[split:]
print(f"Train: {len(train_cases)}, Eval: {len(eval_cases)}")

## 2. 定义 DSPy Signature + Module

In [ ]:
class NewsScoring(dspy.Signature):
    """Evaluate the market impact of a financial news article based ONLY on the provided text.
    Do NOT use any knowledge about what happened after the news was published.
    Output a 5-bin probability distribution (integers summing to 100)."""

    news_text: str = dspy.InputField(desc="Financial news article text")
    strong_bear: int = dspy.OutputField(desc="Probability 0-100 for strong bearish impact")
    weak_bear: int = dspy.OutputField(desc="Probability 0-100 for weak bearish impact")
    neutral: int = dspy.OutputField(desc="Probability 0-100 for neutral impact")
    weak_bull: int = dspy.OutputField(desc="Probability 0-100 for weak bullish impact")
    strong_bull: int = dspy.OutputField(desc="Probability 0-100 for strong bullish impact")
    reasoning: str = dspy.OutputField(desc="Reasoning citing specific text from the news")


class NewsScoringModule(dspy.Module):
    def __init__(self):
        self.scorer = dspy.ChainOfThought(NewsScoring)

    def forward(self, news_text: str):
        return self.scorer(news_text=news_text)


scorer = NewsScoringModule()

# 快速测试
test_resp = scorer(news_text=train_cases[0].news.content)
print(f"Test: strong_bear={test_resp.strong_bear}, weak_bear={test_resp.weak_bear}, "
      f"neutral={test_resp.neutral}, weak_bull={test_resp.weak_bull}, strong_bull={test_resp.strong_bull}")
print(f"Reasoning: {test_resp.reasoning[:150]}...")

## 3. 定义优化目标：泄露评估函数

目标函数 = 最小化 L（泄露）同时保持准确率。DSPy metric 返回 0-1 分数，越高越好。

In [ ]:
def extract_distribution(prediction) -> list[float]:
    """从 DSPy prediction 提取归一化概率分布。"""
    try:
        dist = [
            float(prediction.strong_bear),
            float(prediction.weak_bear),
            float(prediction.neutral),
            float(prediction.weak_bull),
            float(prediction.strong_bull),
        ]
    except (ValueError, AttributeError):
        return [0.2] * 5
    total = sum(dist)
    return [d / total for d in dist] if total > 0 else [0.2] * 5


def extract_direction(prediction) -> str:
    dist = extract_distribution(prediction)
    bull, bear = dist[3] + dist[4], dist[0] + dist[1]
    if bull > bear + 0.1:
        return "up"
    elif bear > bull + 0.1:
        return "down"
    return "neutral"


def leakage_metric(example, prediction, trace=None) -> float:
    """DSPy metric: 对单条样本评估泄露程度。

    对该样本的所有反事实变体运行 scorer，计算 PC/CI/IDS，
    返回 1 - L（越高越好，即泄露越低越好）。
    """
    tc_id = example.case_id
    case_variants = variant_map.get(tc_id, {})
    if not case_variants:
        return 0.5  # 无变体，无法评估

    orig_dir = extract_direction(prediction)
    orig_dist = extract_distribution(prediction)
    orig_conf = max(orig_dist)

    cf_dirs, cf_confs, cf_dists = [], [], []
    for vt_name, variant in case_variants.items():
        try:
            cf_pred = scorer(news_text=variant.modified_content)
            cf_dirs.append(extract_direction(cf_pred))
            cf_dist = extract_distribution(cf_pred)
            cf_dists.append(cf_dist)
            cf_confs.append(max(cf_dist))
        except Exception:
            continue

    if not cf_dirs:
        return 0.5

    # 计算指标
    pc = sum(1 for d in cf_dirs if d == orig_dir) / len(cf_dirs)
    consistent = [d == orig_dir for d in cf_dirs]
    ci_pairs = [(orig_conf, c) for c, m in zip(cf_confs, consistent) if m]
    ci = 1.0 - np.mean([abs(o - c) for o, c in ci_pairs]) if ci_pairs else 0.5
    ids = input_dependency_score([orig_dist] * len(cf_dists), cf_dists)

    L = composite_leakage_score(pc, ci, ids)

    # 准确率奖励：方向正确 +0.1
    acc_bonus = 0.1 if orig_dir == example.expected_direction else 0.0

    # 返回 1 - L + bonus，clamp 到 [0, 1]
    score = max(0.0, min(1.0, 1.0 - L + acc_bonus))
    return score

print("Metric function defined")

## 4. 构建 DSPy 训练集

In [ ]:
# 将 TestCase 转为 DSPy Example
from src.masking import mask_year

def make_example(tc: TestCase) -> dspy.Example:
    # 对训练集使用年份掩码（基础防泄露）
    masked_text = mask_year(tc.news.content)
    return dspy.Example(
        news_text=masked_text,
        case_id=tc.id,
        expected_direction=tc.expected_direction.value,
    ).with_inputs("news_text")

trainset = [make_example(tc) for tc in train_cases]
evalset = [make_example(tc) for tc in eval_cases]

print(f"Trainset: {len(trainset)} examples")
print(f"Evalset:  {len(evalset)} examples")
print(f"Sample: {trainset[0].news_text[:100]}...")

## 5. 运行 DSPy 优化器

使用 MIPROv2 搜索最优 prompt。如果 MIPROv2 不可用，回退到 BootstrapFewShotWithRandomSearch。

In [ ]:
# 优化前基线评估
print("Evaluating baseline (before optimization)...")
baseline_scorer = NewsScoringModule()
baseline_evaluate = dspy.Evaluate(devset=evalset, metric=leakage_metric, num_threads=1, display_progress=True)
baseline_score = baseline_evaluate(baseline_scorer)
print(f"Baseline score: {baseline_score:.3f}")

In [ ]:
# 运行优化
try:
    optimizer = dspy.MIPROv2(
        metric=leakage_metric,
        auto="medium",  # medium 平衡搜索深度和 API 成本
    )
    optimized_scorer = optimizer.compile(
        NewsScoringModule(),
        trainset=trainset,
        max_bootstrapped_demos=2,
        max_labeled_demos=2,
    )
    optimizer_name = "MIPROv2"
except Exception as e:
    print(f"MIPROv2 failed ({e}), falling back to BootstrapFewShotWithRandomSearch")
    optimizer = dspy.BootstrapFewShotWithRandomSearch(
        metric=leakage_metric,
        max_bootstrapped_demos=2,
        max_labeled_demos=2,
        num_candidate_programs=6,
    )
    optimized_scorer = optimizer.compile(
        NewsScoringModule(),
        trainset=trainset,
    )
    optimizer_name = "BootstrapFewShotWithRandomSearch"

print(f"Optimization complete (optimizer: {optimizer_name})")

## 6. 优化后评估 + 对比

In [ ]:
# 优化后评估
print("Evaluating optimized scorer...")
optimized_evaluate = dspy.Evaluate(devset=evalset, metric=leakage_metric, num_threads=1, display_progress=True)
optimized_score = optimized_evaluate(optimized_scorer)
print(f"Optimized score: {optimized_score:.3f}")
print(f"Improvement: {optimized_score - baseline_score:+.3f}")

# 逐样本对比
print(f"\n{'='*60}")
print("Per-sample comparison (eval set):")
for ex in evalset:
    base_pred = baseline_scorer(news_text=ex.news_text)
    opt_pred = optimized_scorer(news_text=ex.news_text)
    base_dir = extract_direction(base_pred)
    opt_dir = extract_direction(opt_pred)
    expected = ex.expected_direction
    print(f"  [{ex.case_id}] expected={expected:7s} baseline={base_dir:7s} optimized={opt_dir:7s}")

## 7. 提取优化后的 Prompt

In [ ]:
# 查看 DSPy 优化后生成的 prompt
print("Optimized prompt (inspecting module):")
print("=" * 60)

# DSPy 将优化结果存储在 module 的 predictor 中
for name, param in optimized_scorer.named_parameters():
    print(f"\n--- {name} ---")
    if hasattr(param, 'signature'):
        print(f"Instructions: {param.signature.instructions}")
    if hasattr(param, 'demos') and param.demos:
        print(f"Demos ({len(param.demos)}):")
        for i, demo in enumerate(param.demos):
            print(f"  Demo {i+1}: {str(demo)[:200]}...")

## 8. 与手工策略对比

加载 05 notebook 的结果，将 DSPy 优化结果加入对比。

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

# 在 eval set 上用反事实计算 DSPy 优化后的 PC/CI/IDS
opt_orig_dirs, opt_cf_dirs = [], []
opt_orig_confs, opt_cf_confs = [], []
opt_orig_dists, opt_cf_dists = [], []
opt_correct = []

for tc in eval_cases:
    masked = mask_year(tc.news.content)
    pred = optimized_scorer(news_text=masked)
    d = extract_direction(pred)
    dist = extract_distribution(pred)
    opt_correct.append(d == tc.expected_direction.value)

    for vt_name, variant in variant_map.get(tc.id, {}).items():
        masked_cf = mask_year(variant.modified_content)
        cf_pred = optimized_scorer(news_text=masked_cf)
        cf_d = extract_direction(cf_pred)
        cf_dist = extract_distribution(cf_pred)

        opt_orig_dirs.append(d)
        opt_cf_dirs.append(cf_d)
        opt_orig_confs.append(max(dist))
        opt_cf_confs.append(max(cf_dist))
        opt_orig_dists.append(dist)
        opt_cf_dists.append(cf_dist)

opt_pc = prediction_consistency(opt_orig_dirs, opt_cf_dirs)
consistent = [o == c for o, c in zip(opt_orig_dirs, opt_cf_dirs)]
opt_ci = confidence_invariance(opt_orig_confs, opt_cf_confs, consistent)
opt_ids = input_dependency_score(opt_orig_dists, opt_cf_dists)
opt_L = composite_leakage_score(opt_pc, opt_ci, opt_ids)
opt_acc = sum(opt_correct) / len(opt_correct) if opt_correct else 0

# 加载手工策略结果
mitigation_path = PROJECT_ROOT / 'data' / 'results' / 'mitigation_compare_results.json'
if mitigation_path.exists():
    with open(mitigation_path, 'r', encoding='utf-8') as f:
        manual_results = json.load(f)
    manual_metrics = manual_results["metrics"]
else:
    # 如果 05 还没跑过，用占位数据
    manual_metrics = [
        {"strategy": "baseline", "PC": 0.7, "CI": 0.8, "IDS": 0.1, "L": 0.49, "accuracy": 0.6},
        {"strategy": "thales_v1", "PC": 0.5, "CI": 0.7, "IDS": 0.2, "L": 0.35, "accuracy": 0.55},
        {"strategy": "full_mask", "PC": 0.4, "CI": 0.6, "IDS": 0.3, "L": 0.25, "accuracy": 0.5},
    ]
    print("(05 results not found, using placeholder data)")

# 合并
compare_rows = []
for m in manual_metrics:
    compare_rows.append(m)
compare_rows.append({
    "strategy": f"DSPy ({optimizer_name})",
    "PC": opt_pc, "CI": opt_ci, "IDS": opt_ids, "L": opt_L, "accuracy": opt_acc,
})

compare_df = pd.DataFrame(compare_rows)
print(compare_df[["strategy", "PC", "CI", "IDS", "L", "accuracy"]].to_string(index=False, float_format="%.3f"))

In [ ]:
# 对比柱状图：手工策略 vs DSPy
fig, ax1 = plt.subplots(figsize=(12, 6))

x = range(len(compare_df))
colors = ["#9E9E9E"] * (len(compare_df) - 1) + ["#2196F3"]  # DSPy 用蓝色高亮
ax1.bar(x, compare_df["L"], color=colors, alpha=0.8)
ax1.set_ylabel("综合泄露分数 L (越低越好)")

ax2 = ax1.twinx()
ax2.plot(x, compare_df["accuracy"], "o-", color="#FF5722", linewidth=2, markersize=8)
ax2.set_ylabel("准确率", color="#FF5722")
ax2.set_ylim(0, 1)

ax1.set_xticks(x)
ax1.set_xticklabels(compare_df["strategy"], rotation=45, ha="right")
ax1.set_title("手工策略 vs DSPy 自动优化")

plt.tight_layout()
plt.savefig(str(PROJECT_ROOT / 'data' / 'results' / 'dspy_vs_manual.png'), dpi=150, bbox_inches='tight')
plt.show()

## 9. 保存结果 + 导出最优 Prompt

In [ ]:
# 保存优化后的 module
optimized_path = PROJECT_ROOT / 'data' / 'results' / 'optimized_scorer.json'
optimized_scorer.save(str(optimized_path))
print(f"Optimized module saved to {optimized_path}")

# 保存对比结果
output = {
    "optimizer": optimizer_name,
    "baseline_score": float(baseline_score),
    "optimized_score": float(optimized_score),
    "improvement": float(optimized_score - baseline_score),
    "dspy_metrics": {"PC": opt_pc, "CI": opt_ci, "IDS": opt_ids, "L": opt_L, "accuracy": opt_acc},
    "comparison": compare_rows,
}
output_path = PROJECT_ROOT / 'data' / 'results' / 'prompt_optimization_results.json'
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(output, f, ensure_ascii=False, indent=2, default=str)
print(f"Results saved to {output_path}")

print(f"\n{'='*60}")
print("Summary:")
print(f"  Optimizer: {optimizer_name}")
print(f"  Baseline score (1-L): {baseline_score:.3f}")
print(f"  Optimized score (1-L): {optimized_score:.3f}")
print(f"  DSPy L={opt_L:.3f}, Accuracy={opt_acc:.1%}")
print(f"  Improvement: {optimized_score - baseline_score:+.3f}")
print(f"\nOptimized prompt exported for Thales Prompt Registry integration.")